## Example 1: Basic Date/Time Parsing

In [ ]:
import coards
import datetime

print("Example 1: Basic Date/Time Parsing")

try:
    print("✓ coards package functionality:")
    
    # Show available time constants  
    time_constants = [
        'SECOND', 'MINUTE', 'HOUR', 'DAY', 'WEEK', 'MONTH', 
        'COMMON_YEAR', 'LEAP_YEAR', 'GREGORIAN_YEAR'
    ]
    
    print("✓ Time constants available:")
    for const in time_constants:
        if hasattr(coards, const):
            value = getattr(coards, const)
            print(f"  {const}: {value}")
    
    # Test basic parsing functionality
    print("✓ Testing date parsing:")
    
    # Test with different COARDS time strings
    coards_strings = [
        "days since 1990-01-01 00:00:00",
        "hours since 2000-01-01 12:00:00", 
        "seconds since 1970-01-01 00:00:00",
        "minutes since 2023-06-15 00:00:00"
    ]
    
    for time_str in coards_strings:
        # A COARDS string is "<unit> since <reference>". parse_units() takes the BARE unit and
        # returns seconds per unit; parse_date() returns the reference as seconds since coards.EPOCH.
        unit_name, reference_str = time_str.split(" since ")
        units = coards.parse_units(unit_name)
        reference_date = coards.EPOCH + datetime.timedelta(seconds=coards.parse_date(reference_str))
        print(f"  '{time_str}':")
        print(f"    Units: {unit_name} = {units:g} s")
        print(f"    Reference: {reference_date}")
        assert units == datetime.timedelta(**{unit_name: 1}).total_seconds(), (unit_name, units)
        assert reference_date == datetime.datetime.fromisoformat(reference_str), reference_date
        assert coards.parse(0, time_str) == reference_date, time_str
    
    # Test date formatting
    print("✓ Testing date formatting:")
    
    sample_date = datetime.datetime(2023, 6, 15, 12, 30, 45)
    print(f"  Sample date: {sample_date}")
    
    # Format using coards: format(date, units) is the inverse of parse(value, units)
    format_units = "hours since 2023-06-15 00:00:00"
    formatted = coards.format(sample_date, format_units)
    print(f"  COARDS formatted: {formatted} {format_units}")
    assert formatted == 45045 / 3600, formatted  # 12:30:45 is 12.5125 hours after midnight
    assert coards.parse(formatted, format_units) == sample_date, coards.parse(formatted, format_units)
    
    # Test parsing a date string
    print("✓ Testing date string parsing:")
    
    date_strings = [
        "2023-06-15 12:30:45",
        "1990-01-01 00:00:00",
        "2000-12-31 23:59:59"
    ]
    
    for date_str in date_strings:
        try:
            parsed_date = coards.parse_date(date_str)
            print(f"  '{date_str}' → {parsed_date}")
        except Exception as e:
            print(f"  Error parsing '{date_str}': {e}")
            raise
    
    print("✓ Basic date/time parsing completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"coards example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Time Unit Conversion and Calculation

In [ ]:
import coards
import datetime

print("Example 2: Time Unit Conversion and Calculation")

try:
    print("✓ Testing time unit conversions:")
    
    # Test with a reference date and time values
    reference_date = datetime.datetime(2000, 1, 1, 0, 0, 0)
    print(f"  Reference date: {reference_date}")
    
    # Create a COARDS time string
    units_string = "days since 2000-01-01 00:00:00"
    print(f"  Units string: {units_string}")
    
    # Parse the units: "<unit> since <reference>". parse_units() takes the BARE unit and returns
    # seconds per unit; parse_date() returns the reference as seconds since coards.EPOCH.
    unit_name, reference_str = units_string.split(" since ")
    units = coards.parse_units(unit_name)
    parsed_ref = coards.EPOCH + datetime.timedelta(seconds=coards.parse_date(reference_str))
    print(f"  Parsed units: {unit_name} ({units:g} s each)")
    print(f"  Parsed reference: {parsed_ref}")
    assert units == coards.DAY, units
    assert parsed_ref == reference_date, parsed_ref

    # Test with different time values
    time_values = [0, 1, 30, 365, 1000]

    print("✓ Converting time values to dates:")
    for value in time_values:
        # Calculate the actual date
        result_date = parsed_ref + datetime.timedelta(seconds=value * units)
        print(f"    {value} {unit_name} → {result_date}")
        assert result_date == reference_date + datetime.timedelta(days=value), (value, result_date)
        assert result_date == coards.parse(value, units_string), (value, result_date)
    
    # Test with different unit types
    print("✓ Testing different time units:")
    
    unit_examples = [
        ("seconds since 1970-01-01 00:00:00", [0, 3600, 86400, 31536000]),
        ("hours since 2023-01-01 00:00:00", [0, 1, 24, 168, 8760]),
        ("minutes since 2023-06-15 12:00:00", [0, 60, 1440, 10080])
    ]
    
    for units_str, values in unit_examples:
        print(f"  Testing: {units_str}")

        unit_name, ref_str = units_str.split(" since ")
        unit_seconds = coards.parse_units(unit_name)
        ref_date = coards.EPOCH + datetime.timedelta(seconds=coards.parse_date(ref_str))
        print(f"    Units: {unit_name} ({unit_seconds:g} s each), Reference: {ref_date}")
        assert ref_date == datetime.datetime.fromisoformat(ref_str), ref_date

        for val in values[:3]:  # Test first 3 values
            result = ref_date + datetime.timedelta(seconds=val * unit_seconds)
            print(f"      {val} → {result}")
            assert result == ref_date + datetime.timedelta(**{unit_name: val}), (units_str, val, result)
    
    # Test time constants for calculations
    print("✓ Using time constants for calculations:")
    
    constants = ['SECOND', 'MINUTE', 'HOUR', 'DAY', 'WEEK']
    base_time = datetime.datetime(2023, 6, 15, 12, 0, 0)
    
    print(f"  Base time: {base_time}")
    
    for const in constants:
        if hasattr(coards, const):
            const_value = getattr(coards, const)
            
            # Add the constant value as seconds
            if isinstance(const_value, (int, float)):
                new_time = base_time + datetime.timedelta(seconds=const_value)
                print(f"    +1 {const} ({const_value}s): {new_time}")
    
    print("✓ Time unit conversion and calculation completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"coards example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: COARDS Parser and Formatter Classes

In [ ]:
import coards
import datetime

print("Example 3: COARDS Parser and Formatter Classes")

try:
    print("✓ Testing COARDS Parser class:")
    
    # Create a Parser instance with units
    units_string = "days since 2000-01-01 00:00:00"
    try:
        parser = coards.Parser(units_string)
        print(f"  Parser instance created with units: {units_string}")
        print(f"  Parser: {parser}")
    except Exception as e:
        print(f"  Error creating parser: {e}")
        # Fall back to testing parser functionality conceptually
        print("  Parser requires units string for initialization")
        raise
    
    # Test parsing different time values (numeric): a Parser is CALLABLE, value -> datetime
    time_values = [0, 1, 30, 365, 1000]
    reference = datetime.datetime(2000, 1, 1, 0, 0, 0)

    print("✓ Parsing numeric time values:")
    parser = coards.Parser(units_string)
    for value in time_values:
        parsed_time = parser(value)
        print(f"  {value} → {parsed_time}")
        print(f"    Type: {type(parsed_time)}")
        assert parsed_time == reference + datetime.timedelta(days=value), (value, parsed_time)

    # Test Formatter class: the inverse of Parser, also CALLABLE, datetime -> value in the units
    print("✓ Testing COARDS Formatter class:")

    formatter = coards.Formatter(units_string)
    print(f"  Formatter instance created with units: {units_string}")
    print(f"  Formatter: {formatter}")

    # Test formatting different datetime objects
    test_dates = [
        datetime.datetime(2023, 6, 15, 12, 30, 45),
        datetime.datetime(2000, 1, 1, 0, 0, 0),
        datetime.datetime(1990, 12, 31, 23, 59, 59),
    ]

    print("✓ Formatting datetime objects:")
    for date_obj in test_dates:
        formatted = formatter(date_obj)
        print(f"  {date_obj} → {formatted}")
        expected = (date_obj - reference).total_seconds() / 86400
        assert abs(formatted - expected) < 1e-9, (date_obj, formatted, expected)
        assert parser(formatted) == date_obj, (date_obj, parser(formatted))  # round trip
    
    # Test Converter class: Converter(from_units, to_units) is CALLABLE, value -> value
    if hasattr(coards, 'Converter'):
        print("✓ Testing COARDS Converter class:")

        to_units = "hours since 1970-01-01 00:00:00"
        converter = coards.Converter(units_string, to_units)
        print(f"  Converter instance: {converter} ({units_string} → {to_units})")

        # Test conversion between formats
        test_values = [0, 365, 1000]
        for value in test_values:
            converted = converter(value)
            print(f"  Converted {value} → {converted}")
            expected = (reference + datetime.timedelta(days=value) - datetime.datetime(1970, 1, 1)).total_seconds() / 3600
            assert abs(converted - expected) < 1e-9, (value, converted, expected)
    
    # Test round-trip conversion concepts
    print("✓ Round-trip conversion concepts:")
    
    # Show the concept even if classes don't work as expected
    reference = datetime.datetime(2000, 1, 1, 0, 0, 0)
    test_values = [365, 1000, 8000]  # Days since reference
    
    print("  Conceptual round-trip (days ↔ datetime):")
    for days_value in test_values:
        # Convert days to datetime
        datetime_obj = reference + datetime.timedelta(days=days_value)
        
        # Convert datetime back to days
        delta = datetime_obj - reference
        back_to_days = delta.total_seconds() / 86400
        
        print(f"  {days_value} days → {datetime_obj} → {back_to_days:.1f} days")
        print(f"    Round-trip accurate: {abs(days_value - back_to_days) < 0.001}")
    
    # Test error handling: every invalid string must raise coards.ParserError
    print("✓ Testing error handling:")

    # Test with invalid units strings, each with the message coards reports for it
    invalid_units = [
        ("invalid-units-string", "Invalid date units: invalid-units-string"),
        ("", "Invalid date units: "),
        ("days since invalid-date", "Invalid date: invalid-date"),
        ("unknown-unit since 2000-01-01", "Invalid date units: unknown-unit"),
    ]

    for invalid, expected_message in invalid_units:
        try:
            parser = coards.Parser(invalid)
        except coards.ParserError as e:
            print(f"  ✓ '{invalid}' rejected as expected ({type(e).__name__})")
            assert str(e) == expected_message, (invalid, str(e))
        else:
            raise AssertionError(f"coards.Parser({invalid!r}) should raise ParserError")
    
    print("✓ COARDS Parser and Formatter classes completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"coards example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: NetCDF Time Coordinate Processing

In [ ]:
import coards
import datetime

print("Example 4: NetCDF Time Coordinate Processing")

try:
    print("✓ Simulating NetCDF time coordinate processing:")
    
    # Simulate common NetCDF time coordinate scenarios
    netcdf_scenarios = [
        {
            'units': 'days since 1900-01-01 00:00:00',
            'values': [0, 365, 10958, 44927],  # 1900, 1901, 1930, 2023
            'description': 'Climate data with 1900 reference'
        },
        {
            'units': 'hours since 1970-01-01 00:00:00', 
            'values': [0, 24, 8760, 438000],  # Unix epoch references
            'description': 'Hourly weather data'
        },
        {
            'units': 'seconds since 2000-01-01 00:00:00',
            'values': [0, 3600, 86400, 31536000],  # Y2K reference
            'description': 'High-resolution sensor data'
        }
    ]
    
    for scenario in netcdf_scenarios:
        print(f"✓ Processing: {scenario['description']}")
        print(f"  Units: {scenario['units']}")

        # Parse the time units: "<unit> since <reference>"; a Parser maps value -> datetime
        unit_name, reference_str = scenario['units'].split(" since ")
        parser = coards.Parser(scenario['units'])
        reference = parser(0)
        print(f"  Reference date: {reference}")
        print(f"  Time unit: {unit_name} ({parser.units:g} s each)")
        assert reference == datetime.datetime.fromisoformat(reference_str), reference

        # Convert numeric values to actual dates
        print("  Time coordinate values:")
        for value in scenario['values']:
            actual_date = parser(value)
            print(f"    {value:8d} → {actual_date}")
            assert actual_date == reference + datetime.timedelta(**{unit_name: value}), (value, actual_date)

        print()
    
    # Test time bounds processing
    print("✓ Processing time bounds (common in climate data):")
    
    bounds_example = {
        'units': 'days since 1950-01-01 00:00:00',
        'time_values': [15, 45, 75],  # Mid-month values
        'time_bounds': [(0, 31), (31, 59), (59, 90)]  # Month boundaries
    }
    
    parser = coards.Parser(bounds_example['units'])
    reference = parser(0)
    print(f"  Reference: {reference}")
    assert reference == datetime.datetime(1950, 1, 1), reference

    print("  Time coordinate with bounds:")
    durations = []
    for i, (time_val, (start_bound, end_bound)) in enumerate(zip(bounds_example['time_values'], bounds_example['time_bounds'])):

        center_date = parser(time_val)
        start_date = parser(start_bound)
        end_date = parser(end_bound)

        print(f"    Period {i+1}:")
        print(f"      Center: {center_date}")
        print(f"      Start:  {start_date}")
        print(f"      End:    {end_date}")
        print(f"      Duration: {(end_date - start_date).days} days")
        print()
        assert start_date == datetime.datetime(1950, i + 1, 1), start_date
        assert start_date < center_date < end_date, (start_date, center_date, end_date)
        durations.append((end_date - start_date).days)

    # January, February (1950 is not a leap year), March
    assert durations == [31, 28, 31], durations
    
    # Test calendar-specific processing
    print("✓ Testing calendar-aware processing:")
    
    # Show different calendar year constants
    calendar_constants = [
        'COMMON_YEAR', 'LEAP_YEAR', 'GREGORIAN_YEAR', 
        'JULIAN_YEAR', 'TROPICAL_YEAR', 'SIDEREAL_YEAR'
    ]
    
    print("  Available calendar year constants:")
    for const in calendar_constants:
        if hasattr(coards, const):
            value = getattr(coards, const)
            days = value / (24 * 3600) if isinstance(value, (int, float)) else 'N/A'
            print(f"    {const}: {value} seconds ({days:.2f} days)")
    
    # Test leap year handling
    print("✓ Testing leap year considerations:")
    
    leap_years = [2000, 2004, 2020, 2024]
    non_leap_years = [1900, 2001, 2021, 2023]
    
    base_date = datetime.datetime(2000, 1, 1)
    
    print("  Leap year calculations:")
    for year in leap_years + non_leap_years:
        year_start = datetime.datetime(year, 1, 1)
        year_end = datetime.datetime(year, 12, 31, 23, 59, 59)
        duration = year_end - year_start
        
        is_leap = duration.days >= 365
        print(f"    {year}: {duration.days + 1} days ({'leap' if is_leap else 'common'} year)")
    
    print("✓ NetCDF time coordinate processing completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"coards example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Advanced COARDS Applications and Utilities

In [ ]:
import coards
import datetime

print("Example 5: Advanced COARDS Applications and Utilities")

try:
    print("✓ Advanced COARDS utilities and applications:")
    
    # Test udunits compatibility functions
    print("✓ Testing udunits compatibility:")
    
    if hasattr(coards, 'from_udunits') and hasattr(coards, 'to_udunits'):
        # from_udunits(value, units) is coards.parse; to_udunits(date, units) is coards.format
        udunits_examples = [
            ("days since 1900-01-01", 10, datetime.datetime(1900, 1, 11)),
            ("hours since 1970-01-01 00:00:00 UTC", 10, datetime.datetime(1970, 1, 1, 10)),
            ("seconds since 2000-01-01T00:00:00Z", 10, datetime.datetime(2000, 1, 1, 0, 0, 10)),
        ]

        for example, value, expected in udunits_examples:
            # Convert from udunits format
            converted = coards.from_udunits(value, example)
            print(f"  from_udunits({value}, '{example}') → {converted}")
            assert converted == expected, (example, converted)

            # Convert back to udunits format
            back_converted = coards.to_udunits(converted, example)
            print(f"  to_udunits(result, '{example}') → {back_converted}")
            assert back_converted == value, (example, back_converted)
            print()
    
    # Test time series processing
    print("✓ Time series processing example:")
    
    # Simulate a time series dataset
    base_units = "days since 2023-01-01 00:00:00"
    time_values = list(range(0, 365, 30))  # Monthly values for a year
    
    parser = coards.Parser(base_units)
    reference = parser(0)
    print(f"  Base units: {base_units}")
    print(f"  Reference: {reference}")
    assert reference == datetime.datetime(2023, 1, 1), reference

    # Convert to actual dates
    dates = [parser(value) for value in time_values]

    print("  Time series dates:")
    for i, (value, date) in enumerate(zip(time_values, dates)):
        print(f"    {i+1:2d}: {value:3d} days → {date.strftime('%Y-%m-%d')}")

    # Calculate time intervals
    print("  Time intervals between points:")
    for i in range(1, len(dates)):
        interval = dates[i] - dates[i-1]
        print(f"    {i}: {interval.days} days")
        assert interval == datetime.timedelta(days=30), (i, interval)
    
    # Test different temporal resolutions
    print("✓ Multi-resolution temporal processing:")
    
    resolutions = [
        ('annual', 'days since 2000-01-01', [0, 365, 730, 1095]),
        ('monthly', 'days since 2023-01-01', [0, 31, 59, 90, 120]),
        ('daily', 'hours since 2023-06-01', [0, 24, 48, 72, 96]),
        ('hourly', 'minutes since 2023-06-15 00:00', [0, 60, 120, 180, 240])
    ]
    
    for res_name, units_str, values in resolutions:
        print(f"  {res_name.title()} resolution:")

        unit_name, reference_str = units_str.split(" since ")
        parser = coards.Parser(units_str)
        reference = parser(0)
        assert reference == datetime.datetime.fromisoformat(reference_str), (units_str, reference)

        for value in values[:3]:  # Show first 3 values
            date = parser(value)
            print(f"    {value:4d} {unit_name} → {date}")
            assert date == reference + datetime.timedelta(**{unit_name: value}), (units_str, value, date)

        print()
    
    # Test specialized time constants
    print("✓ Specialized time constants and calculations:")
    
    special_constants = [
        'JIFFY', 'SHAKE', 'FORTNIGHT', 'LUNAR_MONTH', 
        'TROPICAL_MONTH', 'SIDEREAL_DAY', 'SIDEREAL_HOUR'
    ]
    
    print("  Specialized time units:")
    for const in special_constants:
        if hasattr(coards, const):
            value = getattr(coards, const)
            if isinstance(value, (int, float)):
                if value < 1:
                    unit_desc = f"{value*1000:.3f} milliseconds"
                elif value < 60:
                    unit_desc = f"{value:.3f} seconds"
                elif value < 3600:
                    unit_desc = f"{value/60:.3f} minutes"
                elif value < 86400:
                    unit_desc = f"{value/3600:.3f} hours"
                else:
                    unit_desc = f"{value/86400:.3f} days"
                
                print(f"    {const}: {unit_desc}")
    
    # Test practical applications
    print("✓ Practical COARDS applications:")
    
    applications = [
        "Climate model output processing",
        "Satellite data time coordinate handling", 
        "Weather forecast time series",
        "Ocean observation data",
        "Atmospheric measurement timestamps",
        "Geophysical survey timing",
        "Environmental monitoring systems"
    ]
    
    print("  Common use cases:")
    for app in applications:
        print(f"    - {app}")
    
    # Show integration with other libraries
    print("✓ Integration patterns:")
    
    integrations = [
        "NetCDF4-Python: Time coordinate processing",
        "xarray: Time dimension handling", 
        "pandas: Datetime index conversion",
        "matplotlib: Time axis formatting",
        "scipy: Scientific data timestamps",
        "numpy: Time array calculations"
    ]
    
    for integration in integrations:
        print(f"    - {integration}")
    
    # Test error handling and edge cases: each malformed string must raise its specific error
    print("✓ Error handling and edge cases:")

    edge_cases = [
        ("Invalid units", "invalid units string", coards.ParserError, "Invalid date units"),
        ("Empty string", "", coards.ParserError, "Invalid date units"),
        ("No reference date", "days since", coards.ParserError, "Invalid date units"),
        # month 13 matches coards' date pattern, so datetime itself rejects it
        ("Invalid date", "days since 2023-13-45", ValueError, "month must be in 1..12"),
    ]

    for case_name, case_input, expected_error, expected_message in edge_cases:
        try:
            result = coards.Parser(case_input)
        except expected_error as e:
            print(f"    ✓ {case_name}: rejected as expected ({type(e).__name__})")
            assert expected_message in str(e), (case_name, str(e))
        else:
            raise AssertionError(f"{case_name}: coards.Parser({case_input!r}) should raise {expected_error.__name__}")

    # A reference date in the future is legal COARDS, not an error
    future_year = datetime.datetime.now().year + 100
    result = coards.Parser(f"days since {future_year}-01-01")
    print(f"    ✓ Future date: parsed, reference {result(0)}")
    assert result(0) == datetime.datetime(future_year, 1, 1), result(0)
    
    print("✓ Advanced COARDS applications and utilities completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"coards example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")